# 연습문제 3-3 ~ 3-4 풀이: 모델 요약 정보 읽기, 편향의 역할

3-2절 연습문제 3-3 ~ 3-4의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch03/03-02_example.ipynb`를 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary

## 사전 준비

본문 [코드 3-1]의 `MLP` 클래스를 가져온다.
이후 문제에서 쓸 수 있도록 은닉층의 뉴런 수(`hidden_size`)와 편향 사용 여부(`bias`)를 인자로 받게 했다.
기본값(뉴런 2개, 편향 사용)으로 만들면 본문의 모델과 같다.

In [2]:
class MLP(nn.Module):                               # 본문 [코드 3-1]
    def __init__(self, hidden_size=2, bias=True):
        super().__init__()
        self.hidden_layer = nn.Linear(2, hidden_size, bias=bias)
        self.output_layer = nn.Linear(hidden_size, 1, bias=bias)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.hidden_layer(x))
        return self.activation(self.output_layer(x))

## 연습문제 3-3

> [코드 3-3]의 모델 요약 정보 출력에서 `Param #`은 해당 계층의 파라미터의 수를 나타낸다.
> 시그모이드 활성화 계층을 포함한 네 개의 계층 객체마다 파라미터의 수가 어떻게 결정되었는지 설명해 보자.

**모델 요약 정보 출력하기**
- 입력 (4, 2) 형태로 `summary()`를 호출: [코드 3-3]과 같은 출력

In [3]:
model = MLP()
summary(model, input_size=(4, 2), device='cpu')

Layer (type:depth-idx)                   Output Shape              Param #
MLP                                      [4, 1]                    --
├─Linear: 1-1                            [4, 2]                    6
├─Sigmoid: 1-2                           [4, 2]                    --
├─Linear: 1-3                            [4, 1]                    3
├─Sigmoid: 1-4                           [4, 1]                    --
Total params: 9
Trainable params: 9
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**결과 분석: Linear 계층은 6개와 3개, Sigmoid 계층은 `--`(없음)**
- 첫 번째 Linear(은닉층, 입력 2개, 뉴런 2개): 6개
- 두 번째 Linear(출력층, 입력 2개, 뉴런 1개): 3개
- 두 Sigmoid 계층: `--`로 표시, 학습할 파라미터가 없음
- 전체 파라미터는 6 + 3 = 9개

**파라미터 이름과 형태로 숫자의 근거 확인하기**
- `named_parameters()`로 파라미터의 이름과 형태를 출력
    - `parameters()`는 파라미터 텐서만 차례로 돌려주고, `named_parameters()`는 `hidden_layer.weight`처럼 이름과 텐서를 쌍으로 돌려줌
    - 문제의 풀이와는 무관하고, 숫자의 근거를 눈으로 확인하기 위해 사용
- 가중치는 (뉴런 수, 입력 수) 형태, 편향은 (뉴런 수,) 형태

In [4]:
for name, param in model.named_parameters():
    print(f'{name:22} {str(tuple(param.shape)):8} {param.numel()}개')

hidden_layer.weight    (2, 2)   4개
hidden_layer.bias      (2,)     2개
output_layer.weight    (1, 2)   2개
output_layer.bias      (1,)     1개


**결과 확인: 가중치와 편향을 더한 값이 `Param #`**
- 은닉층: 가중치 (2, 2) = 4개, 편향 (2,) = 2개: 4 + 2 = 6
- 출력층: 가중치 (1, 2) = 2개, 편향 (1,) = 1개: 2 + 1 = 3
- 시그모이드 계층의 파라미터는 목록에 나타나지 않음: 입력을 정해진 식으로 바꿀 뿐이라 학습으로 바뀌는 값이 없음
- 계층의 파라미터 수 = (입력 수 * 뉴런 수) + 뉴런 수

## 연습문제 3-4

> 선형 계층 객체를 만들 때 `bias=False` 인자를 지정하면 편향 없이 가중치만 파라미터로 사용하는 선형 계층이 된다.
> [코드 3-1]에서 정의한 모델 클래스를 수정해 은닉층과 출력층의 편향 파라미터를 제거한 후 모델의 학습 결과를 확인해 보자.

**학습 준비: 편향을 제거한 모델 만들기**
- `MLP` 클래스에서 `bias` 인자를 `False`로 지정하면(`MLP(bias=False)`) 편향을 제거한 모델 생성
- 0.5를 문턱값으로 네 샘플을 모두 바르게 분류하면 성공으로 보고 시드 10개로 비교
- 학습 환경은 본문과 같음: Adam, 학습률 0.01, 평균제곱오차 손실, 1,000 에포크

In [5]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])    # (4, 2): 입력
Y_true = torch.tensor([[0.], [1.], [1.], [0.]])               # (4, 1): XOR 정답

def train_quiet(model, epochs=1000, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for _ in range(epochs):
        loss = criterion(model(X), Y_true)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        Y_pred = model(X)
    ok = torch.equal((Y_pred >= 0.5).float(), Y_true)         # 네 샘플을 모두 맞히면 성공
    return loss.item(), ok, Y_pred.flatten().tolist()

In [6]:
# 편향이 있는 모델과 없는 모델을 각각 생성해 학습
for bias in (True, False):
    success = 0
    for seed in range(1, 11):
        torch.manual_seed(seed)                     # 시드마다 다른 초깃값
        loss, ok, out = train_quiet(MLP(bias=bias))
        success += ok
        if not bias:                                # 편향을 제거한 경우만 시드별로 출력
            print(f'시드 {seed:2d}  손실 {loss:.4f}  출력 {[round(v, 2) for v in out]}')
    print(f'편향 {"사용" if bias else "제거"}: 10번 중 성공 {success}번\n')

편향 사용: 10번 중 성공 5번

시드  1  손실 0.1815  출력 [0.44, 0.96, 0.38, 0.39]
시드  2  손실 0.1301  출력 [0.1, 0.5, 0.92, 0.5]
시드  3  손실 0.1298  출력 [0.1, 0.5, 0.92, 0.5]
시드  4  손실 0.1311  출력 [0.11, 0.91, 0.5, 0.5]
시드  5  손실 0.1777  출력 [0.59, 0.5, 0.67, 0.04]
시드  6  손실 0.1813  출력 [0.44, 0.96, 0.39, 0.39]
시드  7  손실 0.1299  출력 [0.1, 0.5, 0.92, 0.5]
시드  8  손실 0.1776  출력 [0.59, 0.5, 0.67, 0.04]
시드  9  손실 0.1342  출력 [0.14, 0.89, 0.5, 0.5]
시드 10  손실 0.1301  출력 [0.1, 0.92, 0.5, 0.5]
편향 제거: 10번 중 성공 0번



**결과 분석: 편향을 제거하면 10번 모두 실패**
- 편향을 쓰면 10번 중 5번 성공: 뉴런이 두 개뿐이라 편향이 있어도 초깃값에 따라 실패하기도 함
- 편향을 제거하면 10번 모두 실패하고 손실이 0.13 ~ 0.18 근처에서 멈춤
- 네 출력 중 하나 이상이 정답과 반대이거나 0.5 근처에 머무름

**원인 찾기: 입력 (0, 0)의 은닉층 출력**
- 편향이 없으면 뉴런의 가중합은 가중치 * 입력뿐이라 결정 경계가 반드시 원점을 지나는 직선
- 입력 (0, 0)은 원점 자체라 가중치와 상관없이 가중합이 항상 0이고, 시그모이드를 거치면 0.5
- 학습 전과 학습 후에 (0, 0)의 은닉층 출력이 어떻게 달라지는지 확인

In [7]:
torch.manual_seed(1)
model = MLP(bias=False)
origin = torch.tensor([[0., 0.]])
with torch.no_grad():
    print('학습 전:', model.activation(model.hidden_layer(origin)).flatten().tolist())
train_quiet(model)
with torch.no_grad():
    print('학습 후:', model.activation(model.hidden_layer(origin)).flatten().tolist())

학습 전: [0.5, 0.5]
학습 후: [0.5, 0.5]


**결과 확인: 학습해도 (0, 0)의 은닉층 출력은 (0.5, 0.5)로 고정**
- 두 뉴런의 결정 경계가 모두 원점을 지나므로 (0, 0)은 항상 경계 위에 놓임
- 은닉층이 (0, 0)을 한쪽으로 밀어 줄 수 없고, 나머지 세 점의 요구와 맞추기 어려움
- 편향은 결정 경계를 원점에서 옮기는 역할: 이 데이터에는 원점에 있는 샘플이 있어 영향이 큼

**직접 해 보기: 은닉층의 뉴런을 5개로 늘리면?**
- 편향은 그대로 제거하고 `MLP(hidden_size=5, bias=False)`로 시드 10개를 학습
- 뉴런이 많아져도 여전히 모두 실패할까?

In [8]:
success = 0
for seed in range(1, 11):
    torch.manual_seed(seed)
    loss, ok, out = train_quiet(MLP(hidden_size=5, bias=False))
    success += ok
print(f'뉴런 5개, 편향 제거: 10번 중 성공 {success}번')

뉴런 5개, 편향 제거: 10번 중 성공 10번


**결과 확인: 뉴런이 5개이면 편향이 없어도 10번 모두 성공**
- 뉴런이 늘면 은닉층이 만드는 결정 경계가 늘고, 은닉 공간도 2차원에서 5차원으로 넓어짐
- 경계가 모두 원점을 지나더라도 여러 개를 조합하면 네 점을 가를 수 있음
- 편향이 꼭 있어야만 풀리는 것은 아님: 뉴런이 적은 작은 모델일수록 편향의 역할이 중요함